In [0]:
# =============================================================
# logging_utils
# Shared utility functions for pipeline audit logging
# =============================================================
import uuid
from datetime import datetime
from pyspark.sql import Row

# Table reference
log_table = "workspace.ai_project.processing_log"

In [0]:
# Creates the processing_log Delta table if it doesn't exist.

def initialize_log_table():
    if not spark.catalog.tableExists(log_table):
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {log_table} (
                log_id               STRING NOT NULL,
                file_name            STRING,
                source_type          STRING,
                processed_timestamp  TIMESTAMP,
                target_table         STRING,
                chunk_count          INT,
                status               STRING,
                error_message        STRING,
                CONSTRAINT processing_log_pk PRIMARY KEY (log_id)
            )
            USING DELTA
            COMMENT 'Audit log tracking file processing status across all ingestion pipelines'
        """)
        print(f"✨ Created log table: {log_table}")
    else:
        print(f"✅ Log table exists: {log_table}")

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, IntegerType

def write_log(file_name, source_type, target_table, status, chunk_count=0, error_message=None):
    """
    Writes a single log entry to the processing_log Delta table.

    Args:
        file_name       (str): Name of the file being processed
        source_type     (str): Source domain e.g. 'books' or 'docs'
        target_table    (str): Target Delta table the chunks were written to
        status          (str): 'SUCCESS' or 'FAILED'
        chunk_count     (int): Number of chunks written (0 on failure)
        error_message   (str): Error details if status is 'FAILED', else None
    """
    schema = StructType([
        StructField("log_id", StringType(), False),
        StructField("file_name", StringType(), True),
        StructField("source_type", StringType(), True),
        StructField("processed_timestamp", TimestampType(), True),
        StructField("target_table", StringType(), True),
        StructField("chunk_count", IntegerType(), True),
        StructField("status", StringType(), True),
        StructField("error_message", StringType(), True)
    ])

    log_entry = Row(
        log_id=str(uuid.uuid4()),
        file_name=file_name,
        source_type=source_type,
        processed_timestamp=datetime.now(),
        target_table=target_table,
        chunk_count=chunk_count,
        status=status,
        error_message=error_message
    )

    log_df = spark.createDataFrame([log_entry], schema=schema)
    log_df.write.format("delta").mode("append").saveAsTable(log_table)

In [0]:
# Auto-initialize on %run
initialize_log_table()